In [14]:
import pandas as pd

mises_citations_df = pd.read_csv("../data/processed/mises_citations.csv")
hayek_citations_df = pd.read_csv("../data/processed/hayek_citations.csv")
citations_df = pd.read_csv("../data/processed/citations.csv")


In [15]:
hayek_citations_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6281 entries, 0 to 6280
Data columns (total 21 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   Unnamed: 0            6281 non-null   int64  
 1   paper_id              6281 non-null   int64  
 2   raw                   6281 non-null   object 
 3   context               6281 non-null   object 
 4   co_cited_count        6281 non-null   int64  
 5   section_id            6133 non-null   object 
 6   paragraph_id          6281 non-null   object 
 7   sentence_id           6281 non-null   object 
 8   sentence_seq_number   6281 non-null   int64  
 9   reference_seq_number  6281 non-null   int64  
 10  author                6281 non-null   object 
 11  page                  3124 non-null   float64
 12  year                  5768 non-null   float64
 13  title                 6281 non-null   object 
 14  filename              6281 non-null   object 
 15  sentence_count       

In [16]:
import pandas as pd

def build_table(mises_df, hayek_df, citations_df, unit_col):
    
    # Conjuntos principais
    mises_set = set(mises_df[unit_col].dropna().unique())
    hayek_set = set(hayek_df[unit_col].dropna().unique())
    
    full_set = set(citations_df[unit_col].dropna().unique())
    
    # Outros autores (exclui Mises e Hayek)
    others_df = citations_df[
        ~citations_df['author'].isin(['Mises', 'Hayek'])
    ]
    others_set = set(others_df[unit_col].dropna().unique())
    
    # Complementos
    not_mises = full_set - mises_set
    not_hayek = full_set - hayek_set
    
    # Função auxiliar segura
    def safe_div(num, den):
        return num / den if den > 0 else 0
    
    # Probabilidades condicionais
    results = {
        'P(Hayek | Mises)': safe_div(len(mises_set & hayek_set), len(mises_set)),
        'P(Mises | Hayek)': safe_div(len(mises_set & hayek_set), len(hayek_set)),
        
        'P(Outros | Mises)': safe_div(len(mises_set & others_set), len(mises_set)),
        'P(Outros | Hayek)': safe_div(len(hayek_set & others_set), len(hayek_set)),
        
        'P(Mises | Outros)': safe_div(len(mises_set & others_set), len(others_set)),
        'P(Hayek | Outros)': safe_div(len(hayek_set & others_set), len(others_set)),
        
        'P(Hayek | ¬Mises)': safe_div(len(hayek_set & not_mises), len(not_mises)),
        'P(Mises | ¬Hayek)': safe_div(len(mises_set & not_hayek), len(not_hayek)),
        
        'P(Outros | ¬Mises)': safe_div(len(others_set & not_mises), len(not_mises)),
        'P(Outros | ¬Hayek)': safe_div(len(others_set & not_hayek), len(not_hayek)),
    }
    
    table = pd.DataFrame({
        'Probabilidade': list(results.keys()),
        'Valor': list(results.values())
    })
    
    # 🔥 Ordena da maior para a menor
    table = table.sort_values(by='Valor', ascending=False).reset_index(drop=True)
    
    return table


In [17]:
sentence_table = build_table(
    mises_citations_df,
    hayek_citations_df,
    citations_df,
    'sentence_id'
)

print('Sentence-level probabilities')
sentence_table


Sentence-level probabilities


,Probabilidade,Valor
0,P(Outros | ¬Mises),0.976515
1,P(Outros | ¬Hayek),0.970488
2,P(Outros | Hayek),0.263824
3,P(Outros | Mises),0.253837
4,P(Mises | Hayek),0.095833
5,P(Hayek | Mises),0.077529
6,P(Mises | ¬Hayek),0.038051
7,P(Hayek | ¬Mises),0.030412
8,P(Mises | Outros),0.010692
9,P(Hayek | Outros),0.008990


In [18]:
paragraph_table = build_table(
    mises_citations_df,
    hayek_citations_df,
    citations_df,
    'paragraph_id'
)

print('Paragraph-level probabilities')
paragraph_table


Paragraph-level probabilities


,Probabilidade,Valor
0,P(Outros | ¬Mises),0.982882
1,P(Outros | ¬Hayek),0.977834
2,P(Outros | Hayek),0.643593
3,P(Outros | Mises),0.630948
4,P(Mises | Hayek),0.193449
5,P(Hayek | Mises),0.156240
6,P(Mises | ¬Hayek),0.054670
7,P(Hayek | ¬Mises),0.042740
8,P(Mises | Outros),0.040416
9,P(Hayek | Outros),0.033296


In [19]:
section_table = build_table(
    mises_citations_df,
    hayek_citations_df,
    citations_df,
    'section_id'
)

print('Section-level probabilities')
section_table

Section-level probabilities


,Probabilidade,Valor
0,P(Outros | ¬Mises),0.995442
1,P(Outros | ¬Hayek),0.993387
2,P(Outros | Hayek),0.956155
3,P(Outros | Mises),0.949568
4,P(Mises | Hayek),0.442496
5,P(Hayek | Mises),0.365561
6,P(Mises | Outros),0.130480
7,P(Hayek | Outros),0.108542
8,P(Mises | ¬Hayek),0.097146
9,P(Hayek | ¬Mises),0.072452


In [20]:
paper_table = build_table(
    mises_citations_df,
    hayek_citations_df,
    citations_df,
    'paper_id'
)

print('Paper-level probabilities')
paper_table

Paper-level probabilities


,Probabilidade,Valor
0,P(Outros | Mises),1.000000
1,P(Outros | Hayek),1.000000
2,P(Outros | ¬Mises),1.000000
3,P(Outros | ¬Hayek),1.000000
4,P(Mises | Hayek),0.874903
5,P(Mises | Outros),0.696557
6,P(Hayek | Mises),0.615426
7,P(Mises | ¬Hayek),0.525223
8,P(Hayek | Outros),0.489974
9,P(Hayek | ¬Mises),0.201995


Análise de Keywords

In [29]:
import pandas as pd
import numpy as np

def prepare_keywords(df, author_name):
    df = df.copy()
    
    # converter keywords para lista
    if isinstance(df["keywords"].iloc[0], str):
        df["keywords"] = df["keywords"].str.split(",")
    
    df["keywords"] = df["keywords"].apply(
        lambda lst: [k.strip() for k in lst] if isinstance(lst, list) else []
    )
    
    df = df.explode("keywords")
    df["author"] = author_name
    
    return df[["sentence_id", "keywords", "author"]]

# --- Preparação ---
mises = prepare_keywords(mises_citations_df, "Mises")
hayek = prepare_keywords(hayek_citations_df, "Hayek")

df = pd.concat([mises, hayek], ignore_index=True)

# total de sentence_ids únicos
total_sentences = df["sentence_id"].nunique()

# --- Sentence_ids por keyword e autor ---
grouped = (
    df.groupby(["keywords", "author"])["sentence_id"]
    .apply(set)
    .unstack(fill_value=set())
)

# garantir colunas
for col in ["Mises", "Hayek"]:
    if col not in grouped.columns:
        grouped[col] = grouped.apply(lambda x: set(), axis=1)

# --- contagens reais ---
grouped["n_Mises"] = grouped["Mises"].apply(len)
grouped["n_Hayek"] = grouped["Hayek"].apply(len)

grouped["n_both"] = grouped.apply(
    lambda row: len(row["Mises"].intersection(row["Hayek"])),
    axis=1
)

grouped["n_total_keyword"] = grouped.apply(
    lambda row: len(row["Mises"].union(row["Hayek"])),
    axis=1
)

# --- probabilidades reais ---
grouped["P(Mises)"] = grouped["n_Mises"] / total_sentences
grouped["P(Hayek)"] = grouped["n_Hayek"] / total_sentences

grouped["P(Mises and Hayek)"] = grouped["n_both"] / total_sentences

grouped["P(Mises only)"] = (
    (grouped["n_Mises"] - grouped["n_both"]) / total_sentences
)

grouped["P(Hayek only)"] = (
    (grouped["n_Hayek"] - grouped["n_both"]) / total_sentences
)

# --- probabilidades condicionais reais ---
grouped["P(Mises | Hayek)"] = np.where(
    grouped["n_Hayek"] > 0,
    grouped["n_both"] / grouped["n_Hayek"],
    0
)

grouped["P(Hayek | Mises)"] = np.where(
    grouped["n_Mises"] > 0,
    grouped["n_both"] / grouped["n_Mises"],
    0
)

# --- tabela final ---
keyword_table = (
    grouped[
        [
            "P(Mises)",
            "P(Hayek)",
            "P(Mises and Hayek)",
            "P(Mises only)",
            "P(Hayek only)",
            "P(Mises | Hayek)",
            "P(Hayek | Mises)"
        ]
    ]
    .reset_index()
    .rename(columns={"keywords": "keyword"})
    .sort_values("keyword")
)

keyword_table

author,keyword,P(Mises),P(Hayek),P(Mises and Hayek),P(Mises only),P(Hayek only),P(Mises | Hayek),P(Hayek | Mises)
0,'entrepreneurship'],0.000303,0.000227,0.000076,0.000227,0.000152,0.333333,0.250000
1,['business cycle'],0.009092,0.008864,0.003485,0.005606,0.005379,0.393162,0.383333
2,['entrepreneurship'],0.014547,0.004243,0.002046,0.012501,0.002197,0.482143,0.140625
3,['innovation',0.000227,0.000227,0.000076,0.000152,0.000152,0.333333,0.333333
4,['innovation'],0.001061,0.000682,0.000152,0.000909,0.000530,0.222222,0.142857
5,['knowledge problem'],0.000530,0.002652,0.000303,0.000227,0.002349,0.114286,0.571429
6,['praxeology',0.000076,0.000000,0.000000,0.000076,0.000000,0.000000,0.000000
7,['praxeology'],0.020305,0.001288,0.000227,0.020077,0.001061,0.176471,0.011194
8,[],0.531707,0.449276,0.038488,0.493219,0.410789,0.085666,0.072385


Log-Likelihood

In [22]:
import re

# Load allowed keywords once (do this outside tokenize)
keywords_df = pd.read_csv("keywords_austrian_papers.csv")

# Normalize keywords the same way as tokenize does
allowed_keywords = set()

for kws in keywords_df["Author Keywords"].dropna():
    for kw in str(kws).split(";"):  # assuming keywords are separated by ;
        kw = kw.strip().lower()
        kw = re.sub(r'[^a-zA-Z\s]', ' ', kw)
        allowed_keywords.add(kw)

allowed_keywords

{'phillips curve',
 'cognitive science',
 'merger waves',
 'bertrand competition',
 'individual rights',
 'bank clearinghouse',
 'subjective v  objective probability',
 'corporate control',
 'katallactics',
 'federalism and polycentricity',
 'deficits',
 'intertemporal coordination failure',
 'social ecological systems',
 'labor',
 'knowledge problem',
 'behavioral political economy',
 'local government',
 'business groups',
 'insider information',
 'role of government',
 'no fault divorce',
 'israel kirzner',
 'causal theory of mind',
 'time to build',
 'artistic genius',
 'gdp',
 'modernism',
 'interdependence',
 'social theory',
 'modern art',
 'eudaimonia',
 'economics of the arts',
 'theory and history',
 'austrian approach',
 'cluster',
 'interest',
 'witness',
 'arts',
 'peasant economy',
 'the firm',
 'power as mass phenomenon',
 'karl menger              ',
 'firm governance',
 'proficiency test',
 'capital',
 'prejudice',
 'social ontology',
 'hebb',
 'consumption tax',
 'gud

In [23]:
mises_only_df = mises_citations_df[
    (mises_citations_df['co_cited_count'] == 0)
]

hayek_only_df = hayek_citations_df[
    (hayek_citations_df['co_cited_count'] == 0)
]

mises_hayek_df = mises_citations_df[
    (mises_citations_df['co_cited_count'] > 0)
]


import math
import re
from collections import Counter
import pandas as pd

# -------------------------
# Tokenização simples
# -------------------------
def tokenize(text):
    text = text.lower()
    text = re.sub(r'[^a-zA-Z\s]', ' ', text)

    tokens = text.split()

    return [t for t in tokens if t in allowed_keywords]



# -------------------------
# Agregação por unidade
# -------------------------
def aggregate_text(df, group_col):
    return (
        df.groupby(group_col)['context']
        .apply(lambda x: ' '.join(x))
        .dropna()
    )


# -------------------------
# Construir Counter
# -------------------------
def build_counter(text_series):
    tokens = []
    for text in text_series:
        tokens.extend(tokenize(text))
    return Counter(tokens)


# -------------------------
# LLR binário
# -------------------------
def compute_llr(counter_a, counter_b, label_a, label_b):
    results = []
    
    total_a = sum(counter_a.values())
    total_b = sum(counter_b.values())
    
    vocab = set(counter_a.keys()).union(counter_b.keys())
    
    for word in vocab:
        k1 = counter_a.get(word, 0)
        k2 = counter_b.get(word, 0)
        
        if k1 == 0 and k2 == 0:
            continue
        
        E1 = total_a * (k1 + k2) / (total_a + total_b)
        E2 = total_b * (k1 + k2) / (total_a + total_b)
        
        score = 0
        if k1 > 0:
            score += k1 * math.log(k1 / E1)
        if k2 > 0:
            score += k2 * math.log(k2 / E2)
        
        score *= 2
        
        prop_a = k1 / total_a if total_a > 0 else 0
        prop_b = k2 / total_b if total_b > 0 else 0
        
        dominant = label_a if prop_a > prop_b else label_b
        
        results.append((word, score, dominant))
    
    return pd.DataFrame(results, columns=["word", "llr_score", "dominant_in"])\
             .sort_values("llr_score", ascending=False)


# -------------------------
# Função geral
# -------------------------
def run_pairwise_llr(df_a, df_b, label_a, label_b, group_col):
    
    agg_a = aggregate_text(df_a, group_col)
    agg_b = aggregate_text(df_b, group_col)
    
    counter_a = build_counter(agg_a)
    counter_b = build_counter(agg_b)
    
    return compute_llr(counter_a, counter_b, label_a, label_b)



In [24]:

# ===========================
# 🚀 EXECUÇÃO POR SENTENCE_ID
# ===========================

# 1️⃣ Mises-only vs Hayek-only
llr_mh = run_pairwise_llr(
    mises_only_df,
    hayek_only_df,
    "Mises-only",
    "Hayek-only",
    group_col="sentence_id"
)

# 2️⃣ Mises-only vs Mises+Hayek
llr_mm = run_pairwise_llr(
    mises_only_df,
    mises_hayek_df,
    "Mises-only",
    "Mises+Hayek",
    group_col="sentence_id"
)

# 3️⃣ Hayek-only vs Mises+Hayek
llr_hm = run_pairwise_llr(
    hayek_only_df,
    mises_hayek_df,
    "Hayek-only",
    "Mises+Hayek",
    group_col="sentence_id"
)

print("Mises vs Hayek")
print(llr_mh.head(30))

print("Mises vs Co-cited")
print(llr_mm.head(30))

print("Hayek vs Co-cited")
print(llr_hm.head(30))


Mises vs Hayek
                 word    llr_score dominant_in
134             hayek  8189.771819  Hayek-only
295             mises  7397.727491  Mises-only
183         knowledge   496.862169  Hayek-only
364            action   384.322362  Mises-only
234             rules   319.426399  Hayek-only
136        praxeology   190.579121  Mises-only
236       information   148.655130  Hayek-only
181     individualism   133.530416  Hayek-only
94       entrepreneur   132.734781  Mises-only
270       competition   116.744715  Hayek-only
147       calculation   103.752597  Mises-only
1           discovery   102.809764  Hayek-only
103             money    96.164468  Mises-only
298          exchange    93.783455  Mises-only
7      classification    77.234983  Hayek-only
288       probability    74.474879  Mises-only
29        equilibrium    69.878669  Hayek-only
25              labor    68.792875  Mises-only
283      coordination    67.469981  Hayek-only
293            profit    64.090605  Mises-onl

In [25]:
# ==========================
# 🔥 EXECUÇÃO POR PARAGRAPH_ID
# ==========================

# 1️⃣ Mises-only vs Hayek-only
llr_mh_paragraph = run_pairwise_llr(
    mises_only_df,
    hayek_only_df,
    "Mises-only",
    "Hayek-only",
    group_col="paragraph_id"
)

# 2️⃣ Mises-only vs Mises+Hayek
llr_mm_paragraph = run_pairwise_llr(
    mises_only_df,
    mises_hayek_df,
    "Mises-only",
    "Mises+Hayek",
    group_col="paragraph_id"
)

# 3️⃣ Hayek-only vs Mises+Hayek
llr_hm_paragraph = run_pairwise_llr(
    hayek_only_df,
    mises_hayek_df,
    "Hayek-only",
    "Mises+Hayek",
    group_col="paragraph_id"
)


print("Paragraph Level — Mises vs Hayek")
print(llr_mh_paragraph.head(30))

print("Paragraph Level — Mises vs Co-cited")
print(llr_mm_paragraph.head(30))

print("Paragraph Level — Hayek vs Co-cited")
print(llr_hm_paragraph.head(30))

Paragraph Level — Mises vs Hayek
                 word    llr_score dominant_in
134             hayek  8189.771819  Hayek-only
295             mises  7397.727491  Mises-only
183         knowledge   496.862169  Hayek-only
364            action   384.322362  Mises-only
234             rules   319.426399  Hayek-only
136        praxeology   190.579121  Mises-only
236       information   148.655130  Hayek-only
181     individualism   133.530416  Hayek-only
94       entrepreneur   132.734781  Mises-only
270       competition   116.744715  Hayek-only
147       calculation   103.752597  Mises-only
1           discovery   102.809764  Hayek-only
103             money    96.164468  Mises-only
298          exchange    93.783455  Mises-only
7      classification    77.234983  Hayek-only
288       probability    74.474879  Mises-only
29        equilibrium    69.878669  Hayek-only
25              labor    68.792875  Mises-only
283      coordination    67.469981  Hayek-only
293            profit    64

In [26]:
# ==========================
# 🔥 EXECUÇÃO POR SECTION_ID
# ==========================

# 1️⃣ Mises-only vs Hayek-only
llr_mh_section = run_pairwise_llr(
    mises_only_df,
    hayek_only_df,
    "Mises-only",
    "Hayek-only",
    group_col="section_id"
)

# 2️⃣ Mises-only vs Mises+Hayek
llr_mm_section = run_pairwise_llr(
    mises_only_df,
    mises_hayek_df,
    "Mises-only",
    "Mises+Hayek",
    group_col="section_id"
)

# 3️⃣ Hayek-only vs Mises+Hayek
llr_hm_section = run_pairwise_llr(
    hayek_only_df,
    mises_hayek_df,
    "Hayek-only",
    "Mises+Hayek",
    group_col="section_id"
)


print("Section Level — Mises vs Hayek")
print(llr_mh_section.head(30))

print("Section Level — Mises vs Co-cited")
print(llr_mm_section.head(30))

print("Section Level — Hayek vs Co-cited")
print(llr_hm_section.head(30))

Section Level — Mises vs Hayek
                 word    llr_score dominant_in
134             hayek  8049.470382  Hayek-only
295             mises  7316.909454  Mises-only
183         knowledge   492.693281  Hayek-only
364            action   378.307505  Mises-only
234             rules   314.834572  Hayek-only
136        praxeology   186.418239  Mises-only
236       information   147.148716  Hayek-only
181     individualism   144.414398  Hayek-only
94       entrepreneur   127.870088  Mises-only
270       competition   121.548361  Hayek-only
147       calculation   113.592284  Mises-only
1           discovery   102.714905  Hayek-only
298          exchange    90.417125  Mises-only
103             money    84.553712  Mises-only
7      classification    77.162551  Hayek-only
288       probability    74.561824  Mises-only
29        equilibrium    72.528935  Hayek-only
25              labor    68.886496  Mises-only
283      coordination    67.403149  Hayek-only
293            profit    64.1